## Initialization

In [4]:
# Resolve paths from the project root or any folder inside the project.
from pathlib import Path
import os

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "Deliberation.Rproj").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the project folder or Analyses/.")
os.chdir(project_root)

# imports and config
import pandas as pd
import os
from tqdm import tqdm
import time
from huggingface_hub import InferenceClient
import re
import json
import numpy as np

# LLM client setup
client = InferenceClient(
    provider="together",
    api_key="",
)

# load data
validation_ratings = pd.read_csv("Data/validation_ratings_summary.csv").iloc[:, 1:]
print(f"Loaded {len(validation_ratings)} rows.")
print(validation_ratings.head(2))

# prompt definitions

# unified 4-function prompt
scoring_prompt= """You are an expert in cognitive psychology.

You are given a short chunk of a participant's think-aloud transcription during a reasoning task.

Your task is to rate how strongly this chunk expresses each of the following deliberation functions.

Definitions:

- Response Control: Refers to deliberately inhibiting or suppressing a generated response when it is deemed inappropriate for the given context. It often involves avoiding immediate reaction or blocking a response.

- Response Generation: Refers to actively generating new answers through the deliberate computation or mental simulation of solutions to a given problem. It often involves hypothetical thinking, algorithmic reasoning, or systematically working through steps to find solutions to a problem. 

- Response Justification: Refers to deliberately seeking explicit reasons, arguments, or rationalizations to support or scrutinize generated answers. It entails building explanations that clarify, validate or invalidate one’s judgments. It often involves giving reasons, referencing evidence, or explaining why an option makes sense or not.

- Response Regulation: Refers to deliberately monitoring and managing one’s own cognitive processes by allocating effort and resources strategically. It often involves reflective decisions about continuing or stopping deliberation, adjusting cognitive investment, or evaluating progress and coherence. 

Important Instructions:

- Each score should reflect the extent to which the chunk expresses the function using a continuous scale from 0 (not at all present) to 100 (extremely present).
- These functions are distinct, but not mutually exclusive — a chunk might score highly on multiple functions if they co-occur.
- If the chunk contains no trace of any of these 4 functions, assign 0 to all functions. This is perfectly acceptable.
- If the chunk expresses a completely different kind of function (not captured by the 4 above), do not mention it.
- Do not explain or justify the scores even if the chunk clearly expresses a different kind of function.

Output strictly in this structure (and nothing else):

Response Control: [score between 0 and 100]  
Response Generation: [score between 0 and 100]  
Response Justification: [score between 0 and 100]  
Response Regulation: [score between 0 and 100]"""

# confidence prompt
scoring_prompt_confidence = """You are given a short chunk of a participant's think-aloud reasoning.

Estimate how confident the speaker sounds in this passage — based on how committed they seem to their ideas, how certain they appear, and whether they express doubt or hesitation.

Instructions:
- Give a score between 0 (not confident at all) and 100 (extremely confident).
- If you are uncertain, choose a midpoint score around 50.
- Output only the numeric score."""

# config
LLAMA_MODEL  = "meta-llama/Llama-3.3-70B-Instruct"
TEMPERATURE  = 1
TOP_P        = 1
MAX_TOKENS   = 200
N            = 10
MAX_RETRIES  = 4
WAIT_SECONDS = 60
BASE_OUT_DIR = "Output/Prompt_optimization"
OVERWRITE    = False

# helpers
def parse_deliberation_scores(raw: list[str]) -> list[dict]:
    KEYS = ["control", "generation", "justification", "regulation"]
    pattern = re.compile(r"\d+(?:\.\d+)?")
    parsed = []
    for txt in raw:
        lines = [ln for ln in txt.splitlines() if ln.strip()]
        scores = {}
        for i, key in enumerate(KEYS):
            try:
                num = pattern.search(lines[i]).group()
                scores[key] = float(num)
            except Exception:
                scores[key] = np.nan
        parsed.append(scores)
    return parsed

def parse_single_scores(raw: list[str]) -> list[float]:
    scores = []
    for txt in raw:
        m = re.search(r"\d+(?:\.\d+)?", txt)
        scores.append(float(m.group()) if m else np.nan)
    if len(scores) < N:
        scores.extend([np.nan] * (N - len(scores)))
    return scores

def query_llm(system_prompt, user_prompt, n=N, max_retries=MAX_RETRIES, wait=WAIT_SECONDS, parse_func=None):
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=LLAMA_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                n=n
            )
            raw_outputs = [c.message.content.strip() for c in response.choices]
            parsed = parse_func(raw_outputs)

            # Sanity check
            if isinstance(parsed[0], dict):
                if all(all(np.isnan(v) for v in d.values()) for d in parsed):
                    raise ValueError("All parsed scores are NaN.")
            else:
                if all(np.isnan(score) for score in parsed):
                    raise ValueError("All parsed scores are NaN.")
            return parsed

        except Exception as err:
            print(f"Attempt {attempt}/{max_retries} failed: {err}")
            if attempt < max_retries:
                print(f"Retrying in {wait}s…")
                time.sleep(wait)
            else:
                print("Giving up, returning NaN.")
                if parse_func == parse_single_scores:
                    return [np.nan] * n
                else:
                    return [{k: np.nan for k in ["control", "generation", "justification", "regulation"]}] * n

Loaded 99 rows.
                                       response_text  \
0  All right, I'm thinking through the options an...   
1  As I read through it once, I was just kind of ...   

                                       question_text  \
0  If you have only one match and you walk into a...   
1  A monkey, a squirrel, and a bird are racing to...   

              correct_answer             incorrect_answer  \
0                      Match  Oil lamp / Newspaper / Wood   
1  No banana on coconut tree                   Any animal   

   Confidence_mean_human  Control_mean_human  Generation_mean_human  \
0              65.000000           42.142857              56.571429   
1              81.428571           49.285714              41.428571   

   Justification_mean_human  Regulation_mean_human  
0                 49.285714              38.571429  
1                 40.000000              72.857143  


## Llama-3.3-70B

In [ ]:
EXPERIMENT_ID = "unified_v1"

OUT_PATH = os.path.join(BASE_OUT_DIR, f"{EXPERIMENT_ID}.csv")


# scoring loop
results = []

for idx, row in tqdm(validation_ratings.iterrows(), total=len(validation_ratings), desc="Scoring"):

    rec = {
        "idx": idx,
        "response_text": row.response_text,
        "question_text": row.question_text,
        "correct_answer": row.correct_answer,
        "incorrect_answer": row.incorrect_answer,
    }

    # deliberation 4-function scoring
    user_prompt = (
        f"Here is the question:\n'{row.question_text}'\n\n"
        f"The intuitive incorrect answer is: '{row.incorrect_answer}'\n"
        f"The correct answer is: '{row.correct_answer}'\n\n"
        f"Here is the participant’s reasoning:\n'{row.response_text}'"
    )

    parsed_dicts = query_llm(
        scoring_prompt,
        user_prompt,
        n=N,
        parse_func=parse_deliberation_scores
    )
    for func in ["control", "generation", "justification", "regulation"]:
        scores = [d[func] for d in parsed_dicts]
        rec[f"{func}_scores"] = json.dumps(scores)
        rec[f"{func}_mean"] = np.nanmean(scores)

    # confidence scoring
    conf_scores = query_llm(
        scoring_prompt_confidence,
        row.response_text,
        n=N,
        parse_func=parse_single_scores
    )
    rec["confidence_scores"] = json.dumps(conf_scores)
    rec["confidence_mean"] = np.nanmean(conf_scores)

    results.append(rec)

    if (idx + 1) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        print(f"Checkpoint written at row {idx+1}")

# final save
scored_df = pd.DataFrame(results)
scored_df.to_csv(OUT_PATH, index=False)
print(f"finished scoring {len(scored_df)} rows, written to {OUT_PATH}")

## Individual calls

In [ ]:
# prompt definitions (one per function)

scoring_prompt_control = """You are an expert in cognitive psychology.

You are given a short chunk of a participant's think-aloud transcription during a reasoning task.

Your task is to rate how strongly this chunk expresses the following deliberation function:

**Response Control**  
Refers to deliberately inhibiting or suppressing a generated response when it is deemed inappropriate for the given context. It often involves avoiding immediate reaction or blocking a response.

Instructions:
- Rate from 0 (not at all present) to 100 (extremely present).
- If the chunk shows no sign of this function, assign 0.
- Output only the numeric score.
"""

scoring_prompt_generation = """You are an expert in cognitive psychology.

You are given a short chunk of a participant's think-aloud transcription during a reasoning task.

Your task is to rate how strongly this chunk expresses the following deliberation function:

**Response Generation**  
Refers to actively generating new answers through the deliberate computation or mental simulation of solutions to a given problem. It often involves hypothetical thinking, algorithmic reasoning, or systematically working through steps to find solutions to a problem.

Instructions:
- Rate from 0 (not at all present) to 100 (extremely present).
- If the chunk shows no sign of this function, assign 0.
- Output only the numeric score.
"""

scoring_prompt_justification = """You are an expert in cognitive psychology.

You are given a short chunk of a participant's think-aloud transcription during a reasoning task.

Your task is to rate how strongly this chunk expresses the following deliberation function:

**Response Justification**  
Refers to deliberately seeking explicit reasons, arguments, or rationalizations to support or scrutinize generated answers. It entails building explanations that clarify, validate or invalidate one’s judgments. It often involves giving reasons, referencing evidence, or explaining why an option makes sense or not.

Instructions:
- Rate from 0 (not at all present) to 100 (extremely present).
- If the chunk shows no sign of this function, assign 0.
- Output only the numeric score.
"""

scoring_prompt_regulation = """You are an expert in cognitive psychology.

You are given a short chunk of a participant's think-aloud transcription during a reasoning task.

Your task is to rate how strongly this chunk expresses the following deliberation function:

**Response Regulation**  
Refers to deliberately monitoring and managing one’s own cognitive processes by allocating effort and resources strategically. It often involves reflective decisions about continuing or stopping deliberation, adjusting cognitive investment, or evaluating progress and coherence.

Instructions:
- Rate from 0 (not at all present) to 100 (extremely present).
- If the chunk shows no sign of this function, assign 0.
- Output only the numeric score.
"""

# confidence prompt
scoring_prompt_confidence = """You are given a short chunk of a participant's think-aloud reasoning.

Estimate how confident the speaker sounds in this passage — based on how committed they seem to their ideas, how certain they appear, and whether they express doubt or hesitation.

Instructions:
- Give a score between 0 (not confident at all) and 100 (extremely confident).
- If you are uncertain, choose a midpoint score around 50.
- Output only the numeric score.
"""

In [ ]:
EXPERIMENT_ID = "individual_v1"

OUT_PATH = os.path.join(BASE_OUT_DIR, f"{EXPERIMENT_ID}.csv")

# scoring loop
results = []

for idx, row in tqdm(validation_ratings.iterrows(), total=len(validation_ratings), desc="Scoring"):

    rec = {
        "idx": idx,
        "response_text": row.response_text,
        "question_text": row.question_text,
        "correct_answer": row.correct_answer,
        "incorrect_answer": row.incorrect_answer,
    }

    # shared user prompt with question context
    base_prompt = (
        f"Here is the question:\n'{row.question_text}'\n\n"
        f"The intuitive incorrect answer is: '{row.incorrect_answer}'\n"
        f"The correct answer is: '{row.correct_answer}'\n\n"
        f"Here is the participant’s reasoning:\n'{row.response_text}'"
    )

    # score each deliberation function independently
    func_prompts = {
        "control": scoring_prompt_control,
        "generation": scoring_prompt_generation,
        "justification": scoring_prompt_justification,
        "regulation": scoring_prompt_regulation,
    }

    for func, prompt in func_prompts.items():
        scores = query_llm(prompt, base_prompt, n=N, parse_func=parse_single_scores)
        rec[f"{func}_scores"] = json.dumps(scores)
        rec[f"{func}_mean"] = np.nanmean(scores)

    # confidence scoring
    conf_scores = query_llm(
        scoring_prompt_confidence,
        row.response_text,
        n=N,
        parse_func=parse_single_scores
    )
    rec["confidence_scores"] = json.dumps(conf_scores)
    rec["confidence_mean"] = np.nanmean(conf_scores)

    results.append(rec)

    if (idx + 1) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        print(f"Checkpoint written at row {idx+1}")

# final save
scored_df = pd.DataFrame(results)
scored_df.to_csv(OUT_PATH, index=False)
print(f"finished scoring {len(scored_df)} rows, written to {OUT_PATH}")

In [ ]:
# unified 4-function prompt
scoring_prompt_v2= """You are an expert in cognitive psychology.

You are given a short chunk of a participant's think-aloud transcription during a reasoning task.

Your task is to rate how strongly this chunk expresses each of the following deliberation functions.

Definitions:

- Response Control: Refers to deliberately inhibiting, stopping, or correcting a response judged inappropriate (e.g., self-interruption, “wait/but/actually”, rejecting an impulsive answer). A brief “no/stop” alone is not sufficient unless it clearly halts/corrects an emerging response.

- Response Generation: Refers to actively generating new answers through the deliberate computation or mental simulation of solutions to a given problem. It often involves hypothetical thinking, algorithmic reasoning, or systematically working through steps to find solutions to a problem. 

- Response Justification: Refers to deliberately seeking explicit reasons, arguments, or rationalizations to support or scrutinize generated answers. It entails building explanations that clarify, validate or invalidate one’s judgments. It often involves giving reasons, referencing evidence, or explaining why an option makes sense or not.

- Response Regulation: Refers to deliberately monitoring and managing one’s own cognitive processes by allocating effort and resources strategically. It often involves reflective decisions about continuing or stopping deliberation, adjusting cognitive investment, or evaluating progress and coherence. 

Important Instructions:

- Each score should reflect the extent to which the chunk expresses the function using a continuous scale from 0 (not at all present) to 100 (extremely present).
- These functions are distinct, but not mutually exclusive — a chunk might score highly on multiple functions if they co-occur.
- If the chunk contains no trace of any of these 4 functions, assign 0 to all functions. This is perfectly acceptable.
- If the chunk expresses a completely different kind of function (not captured by the 4 above), do not mention it.
- Do not explain or justify the scores even if the chunk clearly expresses a different kind of function.

Output strictly in this structure (and nothing else):

Response Control: [score between 0 and 100]  
Response Generation: [score between 0 and 100]  
Response Justification: [score between 0 and 100]  
Response Regulation: [score between 0 and 100]"""

In [ ]:
EXPERIMENT_ID = "unified_v2"

OUT_PATH = os.path.join(BASE_OUT_DIR, f"{EXPERIMENT_ID}.csv")


# scoring loop
results = []

for idx, row in tqdm(validation_ratings.iterrows(), total=len(validation_ratings), desc="Scoring"):

    rec = {
        "idx": idx,
        "response_text": row.response_text,
        "question_text": row.question_text,
        "correct_answer": row.correct_answer,
        "incorrect_answer": row.incorrect_answer,
    }

    # deliberation 4-function scoring
    user_prompt = (
        f"Here is the question:\n'{row.question_text}'\n\n"
        f"The intuitive incorrect answer is: '{row.incorrect_answer}'\n"
        f"The correct answer is: '{row.correct_answer}'\n\n"
        f"Here is the participant’s reasoning:\n'{row.response_text}'"
    )

    parsed_dicts = query_llm(
        scoring_prompt_v2,
        user_prompt,
        n=N,
        parse_func=parse_deliberation_scores
    )
    for func in ["control", "generation", "justification", "regulation"]:
        scores = [d[func] for d in parsed_dicts]
        rec[f"{func}_scores"] = json.dumps(scores)
        rec[f"{func}_mean"] = np.nanmean(scores)

    # confidence scoring
    conf_scores = query_llm(
        scoring_prompt_confidence,
        row.response_text,
        n=N,
        parse_func=parse_single_scores
    )
    rec["confidence_scores"] = json.dumps(conf_scores)
    rec["confidence_mean"] = np.nanmean(conf_scores)

    results.append(rec)

    if (idx + 1) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        print(f"Checkpoint written at row {idx+1}")

# final save
scored_df = pd.DataFrame(results)
scored_df.to_csv(OUT_PATH, index=False)
print(f"finished scoring {len(scored_df)} rows, written to {OUT_PATH}")

## GPT 5.1

In [ ]:
from openai import OpenAI

API_KEY = ""
client = OpenAI(api_key=API_KEY)

GPT_MODEL = "gpt-5.1"

TEMPERATURE  = 1
TOP_P        = 1
MAX_TOKENS   = 200
N            = 8
MAX_RETRIES  = 4
WAIT_SECONDS = 60


def query_llm(system_prompt, user_prompt, n=N, max_retries=MAX_RETRIES, wait=WAIT_SECONDS, parse_func=None, reasoning_effort = "none"):
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=GPT_MODEL, 
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                max_completion_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                n=n,
                reasoning_effort=reasoning_effort
            )
            raw_outputs = [c.message.content.strip() for c in response.choices]
            parsed = parse_func(raw_outputs)

            # sanity check
            if isinstance(parsed[0], dict):
                if all(all(np.isnan(v) for v in d.values()) for d in parsed):
                    raise ValueError("All parsed scores are NaN.")
            else:
                if all(np.isnan(score) for score in parsed):
                    raise ValueError("All parsed scores are NaN.")
            return parsed

        except Exception as err:
            print(f"Attempt {attempt}/{max_retries} failed: {err}")
            if attempt < max_retries:
                print(f"Retrying in {wait}s…")
                time.sleep(wait)
            else:
                print("Giving up, returning NaN.")
                if parse_func == parse_single_scores:
                    return [np.nan] * n
                else:
                    return [{k: np.nan for k in ["control", "generation", "justification", "regulation"]}] * n

In [ ]:
EXPERIMENT_ID = "unified_gpt_v2"
OUT_PATH = os.path.join(BASE_OUT_DIR, f"{EXPERIMENT_ID}.csv")


results = []

for idx, row in tqdm(validation_ratings.iterrows(), total=len(validation_ratings), desc="Scoring"):
    rec = {
        "idx": idx,
        "response_text": row.response_text,
        "question_text": row.question_text,
        "correct_answer": row.correct_answer,
        "incorrect_answer": row.incorrect_answer,
    }

    user_prompt = (
        f"Here is the question:\n'{row.question_text}'\n\n"
        f"The intuitive incorrect answer is: '{row.incorrect_answer}'\n"
        f"The correct answer is: '{row.correct_answer}'\n\n"
        f"Here is the participant’s reasoning:\n'{row.response_text}'"
    )

    parsed_dicts = query_llm(
        scoring_prompt_v2,
        user_prompt,
        n=N,
        parse_func=parse_deliberation_scores,
    )

    for func in ["control", "generation", "justification", "regulation"]:
        scores = [d[func] for d in parsed_dicts]
        rec[f"{func}_scores"] = json.dumps(scores)
        rec[f"{func}_mean"] = np.nanmean(scores)

    conf_scores = query_llm(
        scoring_prompt_confidence,
        row.response_text,
        n=N,
        parse_func=parse_single_scores,
    )
    rec["confidence_scores"] = json.dumps(conf_scores)
    rec["confidence_mean"] = np.nanmean(conf_scores)

    results.append(rec)

    if (idx + 1) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        print(f"Checkpoint written at row {idx+1}")

scored_df = pd.DataFrame(results)
scored_df.to_csv(OUT_PATH, index=False)
print(f"finished scoring {len(scored_df)} rows, written to {OUT_PATH}")

In [8]:
# unified 4-function prompt
scoring_prompt_v3 = """You are an expert in cognitive psychology.

You are given a short chunk of a participant's think-aloud transcription during a reasoning task.

Your task is to rate how strongly this chunk expresses each of the following deliberation functions.

Definitions:

- Response Control: Refers to deliberately inhibiting, stopping, or correcting a response judged inappropriate (e.g., self-interruption, “wait/but/actually”, rejecting an impulsive answer). A brief “no/stop” alone is not sufficient unless it clearly halts/corrects an emerging response.

- Response Generation: Refers to actively generating new answers through the deliberate computation or mental simulation of solutions to a given problem. It often involves hypothetical thinking, algorithmic reasoning, or systematically working through steps to find solutions to a problem. 

- Response Justification: Refers to deliberately seeking explicit reasons, arguments, or rationalizations to support or scrutinize generated answers. It entails building explanations that clarify, validate or invalidate one’s judgments. It often involves giving reasons, referencing evidence, or explaining why an option makes sense or not.

- Response Regulation: Refers to deliberately monitoring and managing one’s own cognitive processes by allocating effort and resources strategically. It often involves reflective decisions about continuing or stopping deliberation, adjusting cognitive investment, or evaluating progress and coherence. 

Important Instructions:

- Each score should reflect the extent to which the chunk expresses the function using a continuous scale from 0 (not at all present) to 100 (extremely present).
- These functions are distinct, but not mutually exclusive — a chunk might score highly on multiple functions if they co-occur.
- If the chunk expresses a completely different kind of function (not captured by the 4 above), do not mention it.
- Do not explain or justify the scores even if the chunk clearly expresses a different kind of function.

Output strictly in this structure (and nothing else):

Response Control: [score between 0 and 100]  
Response Generation: [score between 0 and 100]  
Response Justification: [score between 0 and 100]  
Response Regulation: [score between 0 and 100]"""

In [ ]:
EXPERIMENT_ID = "unified_gpt_v3"
OUT_PATH = os.path.join(BASE_OUT_DIR, f"{EXPERIMENT_ID}.csv")


results = []

for idx, row in tqdm(validation_ratings.iterrows(), total=len(validation_ratings), desc="Scoring"):
    rec = {
        "idx": idx,
        "response_text": row.response_text,
        "question_text": row.question_text,
        "correct_answer": row.correct_answer,
        "incorrect_answer": row.incorrect_answer,
    }

    user_prompt = (
        f"Here is the question:\n'{row.question_text}'\n\n"
        f"The intuitive incorrect answer is: '{row.incorrect_answer}'\n"
        f"The correct answer is: '{row.correct_answer}'\n\n"
        f"Here is the participant’s reasoning:\n'{row.response_text}'"
    )

    parsed_dicts = query_llm(
        scoring_prompt_v3,
        user_prompt,
        n=N,
        parse_func=parse_deliberation_scores,
    )

    for func in ["control", "generation", "justification", "regulation"]:
        scores = [d[func] for d in parsed_dicts]
        rec[f"{func}_scores"] = json.dumps(scores)
        rec[f"{func}_mean"] = np.nanmean(scores)

    conf_scores = query_llm(
        scoring_prompt_confidence,
        row.response_text,
        n=N,
        parse_func=parse_single_scores,
    )
    rec["confidence_scores"] = json.dumps(conf_scores)
    rec["confidence_mean"] = np.nanmean(conf_scores)

    results.append(rec)

    if (idx + 1) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        print(f"Checkpoint written at row {idx+1}")

scored_df = pd.DataFrame(results)
scored_df.to_csv(OUT_PATH, index=False)
print(f"finished scoring {len(scored_df)} rows, written to {OUT_PATH}")

In [ ]:
# unified 4-function prompt
scoring_prompt_v4 = """You are an expert in cognitive psychology.

You are given a short chunk of a participant's think-aloud transcription during a reasoning task.

Your task is to rate how strongly this chunk expresses each of the following deliberation functions.

Definitions:

- Response Control: Refers to deliberately inhibiting, stopping, or correcting a response judged inappropriate (e.g., self-interruption, “wait/but/actually”, rejecting an impulsive answer). A brief “no/stop” alone is not sufficient unless it clearly halts/corrects an emerging response.

- Response Generation: Refers to actively generating new answers through the deliberate computation or mental simulation of solutions to a given problem, even if the attempt is only partial or incomplete. It often involves hypothetical thinking, algorithmic reasoning, or systematically working through steps to find solutions to a problem. 

- Response Justification: Refers to deliberately seeking explicit reasons, arguments, or rationalizations to support or scrutinize generated answers. It entails building explanations that clarify, validate or invalidate one’s judgments. It often involves giving reasons, referencing evidence, or explaining why an option makes sense or not.

- Response Regulation: Refers to deliberately monitoring and managing one’s own cognitive processes by allocating effort and resources strategically. It often involves reflective decisions about continuing or stopping deliberation, adjusting cognitive investment, or evaluating progress and coherence. 

Important Instructions:

- Each score should reflect the extent to which the chunk expresses the function using a continuous scale from 0 (not at all present) to 100 (extremely present).
- These functions are distinct, but not mutually exclusive — a chunk might score highly on multiple functions if they co-occur.
- If the chunk expresses a completely different kind of function (not captured by the 4 above), do not mention it.
- Do not explain or justify the scores even if the chunk clearly expresses a different kind of function.

Output strictly in this structure (and nothing else):

Response Control: [score between 0 and 100]  
Response Generation: [score between 0 and 100]  
Response Justification: [score between 0 and 100]  
Response Regulation: [score between 0 and 100]"""

In [ ]:
EXPERIMENT_ID = "unified_gpt_v4"
OUT_PATH = os.path.join(BASE_OUT_DIR, f"{EXPERIMENT_ID}.csv")


results = []

for idx, row in tqdm(validation_ratings.iterrows(), total=len(validation_ratings), desc="Scoring"):
    rec = {
        "idx": idx,
        "response_text": row.response_text,
        "question_text": row.question_text,
        "correct_answer": row.correct_answer,
        "incorrect_answer": row.incorrect_answer,
    }

    user_prompt = (
        f"Here is the question:\n'{row.question_text}'\n\n"
        f"The intuitive incorrect answer is: '{row.incorrect_answer}'\n"
        f"The correct answer is: '{row.correct_answer}'\n\n"
        f"Here is the participant’s reasoning:\n'{row.response_text}'"
    )

    parsed_dicts = query_llm(
        scoring_prompt_v4,
        user_prompt,
        n=N,
        parse_func=parse_deliberation_scores,
    )

    for func in ["control", "generation", "justification", "regulation"]:
        scores = [d[func] for d in parsed_dicts]
        rec[f"{func}_scores"] = json.dumps(scores)
        rec[f"{func}_mean"] = np.nanmean(scores)

    conf_scores = query_llm(
        scoring_prompt_confidence,
        row.response_text,
        n=N,
        parse_func=parse_single_scores,
    )
    rec["confidence_scores"] = json.dumps(conf_scores)
    rec["confidence_mean"] = np.nanmean(conf_scores)

    results.append(rec)

    if (idx + 1) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        print(f"Checkpoint written at row {idx+1}")

scored_df = pd.DataFrame(results)
scored_df.to_csv(OUT_PATH, index=False)
print(f"finished scoring {len(scored_df)} rows, written to {OUT_PATH}")

In [ ]:
# unified 4-function prompt
scoring_prompt_v5 = """You are an expert in cognitive psychology.

You are given a short chunk of a participant's think-aloud transcription during a reasoning task.

Your task is to rate how strongly this chunk expresses each of the following deliberation functions.

Definitions:

- Response Control: Refers to deliberately inhibiting, stopping, or correcting a response judged inappropriate (e.g., self-interruption, “wait/but/actually”, rejecting an impulsive answer). A brief “no/stop” alone is not sufficient unless it clearly halts/corrects an emerging response.

Examples:
    - Low: “It’s X.”
    - Moderate: “It’s X… wait… maybe Y.”
    - High: “X. No, stop. That’s incorrect. It’s Y.”

- Response Generation: Refers to actively generating new answers through the deliberate computation or mental simulation of solutions to a given problem. It often involves hypothetical thinking, algorithmic reasoning, or systematically working through steps to find solutions to a problem. 

Examples:
    - Low: “I have no idea… I’ll guess X.”
    - Moderate: “If A is true, then B… so maybe X.”
    - High: “Step by step: A, then B, which implies C; therefore X.”

- Response Justification: Refers to deliberately seeking explicit reasons, arguments, or rationalizations to support or scrutinize generated answers. It entails building explanations that clarify, validate or invalidate one’s judgments. It often involves giving reasons, referencing evidence, or explaining why an option makes sense or not.

Examples:
    - Low: “X.”
    - Moderate: “X, because A.”
    - High: “X, because A implies B, which rules out Y; also Y would predict C, but it doesn’t.”

- Response Regulation: Refers to deliberately monitoring and managing one’s own cognitive processes by allocating effort and resources strategically. It often involves reflective decisions about continuing or stopping deliberation, adjusting cognitive investment, or evaluating progress and coherence. 

Examples:
    - Low: “I'm just going to guess X.”
    - Moderate: “Let me double-check.”
    - High: “I’ll slow down, re-read, test an alternative, then decide whether to revise or stop.”

Important Instructions:

- Each score should reflect the extent to which the chunk expresses the function using a continuous scale from 0 (not at all present) to 100 (extremely present).
- These functions are distinct, but not mutually exclusive — a chunk might score highly on multiple functions if they co-occur.
- If the chunk expresses a completely different kind of function (not captured by the 4 above), do not mention it.
- Do not explain or justify the scores even if the chunk clearly expresses a different kind of function.

Output strictly in this structure (and nothing else):

Response Control: [score between 0 and 100]  
Response Generation: [score between 0 and 100]  
Response Justification: [score between 0 and 100]  
Response Regulation: [score between 0 and 100]"""

In [ ]:
EXPERIMENT_ID = "unified_gpt_v5"
OUT_PATH = os.path.join(BASE_OUT_DIR, f"{EXPERIMENT_ID}.csv")


results = []

for idx, row in tqdm(validation_ratings.iterrows(), total=len(validation_ratings), desc="Scoring"):
    rec = {
        "idx": idx,
        "response_text": row.response_text,
        "question_text": row.question_text,
        "correct_answer": row.correct_answer,
        "incorrect_answer": row.incorrect_answer,
    }

    user_prompt = (
        f"Here is the question:\n'{row.question_text}'\n\n"
        f"The intuitive incorrect answer is: '{row.incorrect_answer}'\n"
        f"The correct answer is: '{row.correct_answer}'\n\n"
        f"Here is the participant’s reasoning:\n'{row.response_text}'"
    )

    parsed_dicts = query_llm(
        scoring_prompt_v5,
        user_prompt,
        n=N,
        parse_func=parse_deliberation_scores,
    )

    for func in ["control", "generation", "justification", "regulation"]:
        scores = [d[func] for d in parsed_dicts]
        rec[f"{func}_scores"] = json.dumps(scores)
        rec[f"{func}_mean"] = np.nanmean(scores)

    conf_scores = query_llm(
        scoring_prompt_confidence,
        row.response_text,
        n=N,
        parse_func=parse_single_scores,
    )
    rec["confidence_scores"] = json.dumps(conf_scores)
    rec["confidence_mean"] = np.nanmean(conf_scores)

    results.append(rec)

    if (idx + 1) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        print(f"Checkpoint written at row {idx+1}")

scored_df = pd.DataFrame(results)
scored_df.to_csv(OUT_PATH, index=False)
print(f"finished scoring {len(scored_df)} rows, written to {OUT_PATH}")

In [10]:
EXPERIMENT_ID = "unified_gpt_v3_medium_reasoning"
OUT_PATH = os.path.join(BASE_OUT_DIR, f"{EXPERIMENT_ID}.csv")


results = []

for idx, row in tqdm(validation_ratings.iterrows(), total=len(validation_ratings), desc="Scoring"):
    rec = {
        "idx": idx,
        "response_text": row.response_text,
        "question_text": row.question_text,
        "correct_answer": row.correct_answer,
        "incorrect_answer": row.incorrect_answer,
    }

    user_prompt = (
        f"Here is the question:\n'{row.question_text}'\n\n"
        f"The intuitive incorrect answer is: '{row.incorrect_answer}'\n"
        f"The correct answer is: '{row.correct_answer}'\n\n"
        f"Here is the participant’s reasoning:\n'{row.response_text}'"
    )

    parsed_dicts = query_llm(
        scoring_prompt_v3,
        user_prompt,
        n=N,
        parse_func=parse_deliberation_scores,
        reasoning_effort = "medium"
    )

    for func in ["control", "generation", "justification", "regulation"]:
        scores = [d[func] for d in parsed_dicts]
        rec[f"{func}_scores"] = json.dumps(scores)
        rec[f"{func}_mean"] = np.nanmean(scores)

    conf_scores = query_llm(
        scoring_prompt_confidence,
        row.response_text,
        n=N,
        parse_func=parse_single_scores,
    )
    rec["confidence_scores"] = json.dumps(conf_scores)
    rec["confidence_mean"] = np.nanmean(conf_scores)

    results.append(rec)

    if (idx + 1) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        print(f"Checkpoint written at row {idx+1}")

scored_df = pd.DataFrame(results)
scored_df.to_csv(OUT_PATH, index=False)
print(f"finished scoring {len(scored_df)} rows, written to {OUT_PATH}")

Scoring:  20%|██        | 20/99 [01:57<06:36,  5.02s/it]

Checkpoint written at row 20


Scoring:  40%|████      | 40/99 [03:48<05:34,  5.67s/it]

Checkpoint written at row 40


Scoring:  61%|██████    | 60/99 [05:17<02:55,  4.51s/it]

Checkpoint written at row 60


Scoring:  81%|████████  | 80/99 [06:58<01:22,  4.33s/it]

Checkpoint written at row 80


Scoring: 100%|██████████| 99/99 [08:24<00:00,  5.09s/it]

finished scoring 99 rows, written to ../Output/Prompt_optimization/unified_gpt_v3_medium_reasoning.csv


In [10]:
EXPERIMENT_ID = "unified_gpt_v3_low_reasoning"
OUT_PATH = os.path.join(BASE_OUT_DIR, f"{EXPERIMENT_ID}.csv")


results = []

for idx, row in tqdm(validation_ratings.iterrows(), total=len(validation_ratings), desc="Scoring"):
    rec = {
        "idx": idx,
        "response_text": row.response_text,
        "question_text": row.question_text,
        "correct_answer": row.correct_answer,
        "incorrect_answer": row.incorrect_answer,
    }

    user_prompt = (
        f"Here is the question:\n'{row.question_text}'\n\n"
        f"The intuitive incorrect answer is: '{row.incorrect_answer}'\n"
        f"The correct answer is: '{row.correct_answer}'\n\n"
        f"Here is the participant’s reasoning:\n'{row.response_text}'"
    )

    parsed_dicts = query_llm(
        scoring_prompt_v3,
        user_prompt,
        n=N,
        parse_func=parse_deliberation_scores,
        reasoning_effort = "low"
    )

    for func in ["control", "generation", "justification", "regulation"]:
        scores = [d[func] for d in parsed_dicts]
        rec[f"{func}_scores"] = json.dumps(scores)
        rec[f"{func}_mean"] = np.nanmean(scores)

    conf_scores = query_llm(
        scoring_prompt_confidence,
        row.response_text,
        n=N,
        parse_func=parse_single_scores,
    )
    rec["confidence_scores"] = json.dumps(conf_scores)
    rec["confidence_mean"] = np.nanmean(conf_scores)

    results.append(rec)

    if (idx + 1) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        print(f"Checkpoint written at row {idx+1}")

scored_df = pd.DataFrame(results)
scored_df.to_csv(OUT_PATH, index=False)
print(f"finished scoring {len(scored_df)} rows, written to {OUT_PATH}")

Scoring:  20%|██        | 20/99 [01:25<04:29,  3.41s/it]

Checkpoint written at row 20


Scoring:  40%|████      | 40/99 [02:33<03:03,  3.11s/it]

Checkpoint written at row 40


Scoring:  61%|██████    | 60/99 [03:40<02:02,  3.14s/it]

Checkpoint written at row 60


Scoring:  81%|████████  | 80/99 [04:50<01:08,  3.60s/it]

Checkpoint written at row 80


Scoring: 100%|██████████| 99/99 [06:00<00:00,  3.64s/it]

finished scoring 99 rows, written to ../Output/Prompt_optimization/unified_gpt_v3_low_reasoning.csv


In [12]:
EXPERIMENT_ID = "unified_gpt_v3_high_reasoning"
OUT_PATH = os.path.join(BASE_OUT_DIR, f"{EXPERIMENT_ID}.csv")


results = []

for idx, row in tqdm(validation_ratings.iterrows(), total=len(validation_ratings), desc="Scoring"):
    rec = {
        "idx": idx,
        "response_text": row.response_text,
        "question_text": row.question_text,
        "correct_answer": row.correct_answer,
        "incorrect_answer": row.incorrect_answer,
    }

    user_prompt = (
        f"Here is the question:\n'{row.question_text}'\n\n"
        f"The intuitive incorrect answer is: '{row.incorrect_answer}'\n"
        f"The correct answer is: '{row.correct_answer}'\n\n"
        f"Here is the participant’s reasoning:\n'{row.response_text}'"
    )

    parsed_dicts = query_llm(
        scoring_prompt_v3,
        user_prompt,
        n=N,
        parse_func=parse_deliberation_scores,
        reasoning_effort = "high"
    )

    for func in ["control", "generation", "justification", "regulation"]:
        scores = [d[func] for d in parsed_dicts]
        rec[f"{func}_scores"] = json.dumps(scores)
        rec[f"{func}_mean"] = np.nanmean(scores)

    conf_scores = query_llm(
        scoring_prompt_confidence,
        row.response_text,
        n=N,
        parse_func=parse_single_scores,
    )
    rec["confidence_scores"] = json.dumps(conf_scores)
    rec["confidence_mean"] = np.nanmean(conf_scores)

    results.append(rec)

    if (idx + 1) % 20 == 0:
        pd.DataFrame(results).to_csv(OUT_PATH, index=False)
        print(f"Checkpoint written at row {idx+1}")

scored_df = pd.DataFrame(results)
scored_df.to_csv(OUT_PATH, index=False)
print(f"finished scoring {len(scored_df)} rows, written to {OUT_PATH}")

Scoring:   1%|          | 1/99 [00:05<08:52,  5.43s/it]

Attempt 1/4 failed: All parsed scores are NaN.
Retrying in 60s…
Attempt 2/4 failed: All parsed scores are NaN.
Retrying in 60s…


Scoring:   6%|▌         | 6/99 [02:55<25:56, 16.74s/it]  

Attempt 1/4 failed: Error code: 400 - {'error': {'message': 'Could not finish the message because max_tokens or model output limit was reached. Please try again with higher max_tokens.', 'type': 'invalid_request_error', 'param': None, 'code': None}}
Retrying in 60s…


Scoring:  20%|██        | 20/99 [05:22<07:34,  5.76s/it]

...

Checkpoint written at row 80


Scoring:  96%|█████████▌| 95/99 [21:04<00:21,  5.41s/it]

Attempt 1/4 failed: All parsed scores are NaN.
Retrying in 60s…
Attempt 2/4 failed: All parsed scores are NaN.
Retrying in 60s…
Attempt 3/4 failed: All parsed scores are NaN.
Retrying in 60s…


Scoring: 100%|██████████| 99/99 [24:42<00:00, 14.97s/it]

finished scoring 99 rows, written to ../Output/Prompt_optimization/unified_gpt_v3_high_reasoning.csv
